In [0]:
%pip install fitparse

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def create_moving_ratio_chart(
    df: pd.DataFrame,
    speed_threshold_kmh: float = 0.5,   # Drempel op ruwe data: < 0.5 km/u is echte stilstand
    min_pause_sec: int = 150,           # Enkel pauzes vanaf 2,5 minuut worden rood gearcheerd
    savgol_window: int = 301,           # 5 minuten (301s) venster voor een zeer vloeiende lijn
    savgol_polyorder: int = 2
):
    df_clean = df.copy()

    # 0a. Bepaal dynamisch de tijdkolom
    if 'time' in df_clean.columns:
        time_col = 'time'
    elif 'timestamp' in df_clean.columns:
        time_col = 'timestamp'
    else:
        raise KeyError("Het DataFrame bevat geen 'time' of 'timestamp' kolom.")

    df_clean[time_col] = pd.to_datetime(df_clean[time_col])
    df_clean = df_clean.sort_values(time_col).reset_index(drop=True)

    # 0b. Bepaal dynamisch de snelheid (in km/u)
    if 'speed_ms' in df_clean.columns:
        df_clean['raw_speed_kmh'] = df_clean['speed_ms'] * 3.6
    elif 'speed' in df_clean.columns:
        df_clean['raw_speed_kmh'] = df_clean['speed'] * 3.6
    elif 'enhanced_speed' in df_clean.columns:
        df_clean['raw_speed_kmh'] = df_clean['enhanced_speed'] * 3.6
    elif 'distance' in df_clean.columns:
        time_diff = df_clean[time_col].diff().dt.total_seconds().fillna(0)
        dist_diff = df_clean['distance'].diff().fillna(0)
        speed_ms = np.where(time_diff > 0, dist_diff / time_diff, 0)
        df_clean['raw_speed_kmh'] = speed_ms * 3.6
    else:
        raise KeyError("Geen kolom gevonden voor snelheid.")

    # 0c. Bepaal dynamisch de hoogtekolom
    alt_col = None
    for candidate in ['altitude', 'enhanced_altitude', 'ele', 'elevation']:
        if candidate in df_clean.columns:
            alt_col = candidate
            break

    # --- 1. DETECTEER GARMIN PAUZES & HERCONSTRUEER CONTINU TIDSGRID ---
    df_clean['time_diff_sec'] = df_clean[time_col].diff().dt.total_seconds().fillna(0)

    # Maak een continue 1-seconde tijdlijn aan
    df_clean = df_clean.set_index(time_col)
    df_resampled = pd.DataFrame(index=pd.date_range(start=df_clean.index.min(), end=df_clean.index.max(), freq='1s'))
    df_resampled['speed_kmh'] = 0.0

    # Vul snelheden in op meetpunten; gaten (>15s) blijven 0.0 km/u
    for i in range(len(df_clean)):
        t_curr = df_clean.index[i]
        t_diff = df_clean['time_diff_sec'].iloc[i]
        spd = df_clean['raw_speed_kmh'].iloc[i]
        
        if t_diff <= 15:
            t_prev = t_curr - pd.Timedelta(seconds=int(t_diff))
            df_resampled.loc[t_prev:t_curr, 'speed_kmh'] = spd
        else:
            df_resampled.loc[t_curr, 'speed_kmh'] = spd

    if alt_col is not None:
        df_resampled[alt_col] = df_clean[alt_col].resample('1s').ffill().bfill()

    df_resampled = df_resampled.reset_index().rename(columns={'index': time_col})

    # 2. Verstreken tijd in minuten
    start_time = df_resampled[time_col].min()
    df_resampled['elapsed_min'] = (df_resampled[time_col] - start_time).dt.total_seconds() / 60.0

    # --- 3. PAUZEDETECTIE OP RUWE DATA (VOORKOMT FOUTIEVE PAUZES) ---
    df_resampled['is_raw_idle'] = df_resampled['speed_kmh'] < speed_threshold_kmh
    df_resampled['pause_group'] = (~df_resampled['is_raw_idle']).cumsum()
    pause_durations = df_resampled[df_resampled['is_raw_idle']].groupby('pause_group')['is_raw_idle'].transform('count')
    
    # Alleen aaneengesloten stilstand van minstens 150s (2.5 min) is een geldige rustpauze
    df_resampled['is_valid_pause'] = df_resampled['is_raw_idle'] & (pause_durations >= min_pause_sec)

    # --- 4. 5-MINUTEN SMOOTHING VOOR VISUALISATIE ---
    if savgol_window % 2 == 0:
        savgol_window += 1
    actual_window = min(savgol_window, len(df_resampled) if len(df_resampled) % 2 != 0 else len(df_resampled) - 1)

    if len(df_resampled) >= actual_window and actual_window > savgol_polyorder:
        df_resampled['speed_kmh_smooth'] = savgol_filter(
            df_resampled['speed_kmh'], 
            window_length=actual_window, 
            polyorder=savgol_polyorder
        )
        df_resampled['speed_kmh_smooth'] = df_resampled['speed_kmh_smooth'].clip(lower=0)
    else:
        df_resampled['speed_kmh_smooth'] = df_resampled['speed_kmh']

    # Zet snelheid tijdens vastgestelde pauzes exact op 0 km/u
    df_resampled.loc[df_resampled['is_valid_pause'], 'speed_kmh_smooth'] = 0.0

    # 5. Exacte tijdberekening
    total_time_sec = (df_resampled[time_col].max() - start_time).total_seconds()
    idle_time_sec = df_resampled['is_valid_pause'].sum()
    moving_time_sec = total_time_sec - idle_time_sec
    moving_ratio = (moving_time_sec / total_time_sec) * 100 if total_time_sec > 0 else 0

    def format_time(sec):
        hrs = int(sec // 3600)
        mins = int((sec % 3600) // 60)
        return f"{hrs}u {mins:02d}m" if hrs > 0 else f"{mins}m"

    # --- 6. GRAFIEK OPBOUWEN ---
    plt.style.use('dark_background')
    BG_COLOR = '#0B0F19'
    CARD_BG = '#111827'
    ACCENT_BLUE = '#00D2FF'
    ACCENT_RED = '#FF4B4B'
    ACCENT_GREEN = '#00C853'
    TEXT_MUTED = '#9CA3AF'

    fig = plt.figure(figsize=(14, 7), facecolor=BG_COLOR)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.2, 1], wspace=0.15)

    ax_main = fig.add_subplot(gs[0], facecolor=BG_COLOR)
    ax_donut = fig.add_subplot(gs[1], facecolor=BG_COLOR)

    # --- HOOGTEPROFIEL ---
    ax_alt = ax_main.twinx()
    if alt_col is not None:
        alt_data = df_resampled[alt_col]
        ax_alt.plot(df_resampled['elapsed_min'], alt_data, color='#4A5568', alpha=0.35, linewidth=1.2, label='Hoogte')
        ax_alt.fill_between(df_resampled['elapsed_min'], alt_data, color='#4A5568', alpha=0.12)
        ax_alt.set_ylabel('Hoogte (m)', color=TEXT_MUTED, fontsize=10, labelpad=8)
        ax_alt.tick_params(colors=TEXT_MUTED, labelsize=9)
        ax_alt.grid(False)
        
        alt_min, alt_max = alt_data.min(), alt_data.max()
        alt_pad = (alt_max - alt_min) * 0.2 if alt_max > alt_min else 10
        ax_alt.set_ylim(alt_min - alt_pad, alt_max + alt_pad * 3)
    else:
        ax_alt.axis('off')

    # --- PAUZEVLAKKEN & SNELHEIDSLIJN ---
    max_speed = max(df_resampled['speed_kmh_smooth'].max() * 1.15, 5.0)

    ax_main.fill_between(
        df_resampled['elapsed_min'], 
        0, 
        max_speed,
        where=df_resampled['is_valid_pause'],
        color=ACCENT_RED, 
        alpha=0.28, 
        step='post',
        label='Rustpauze'
    )

    ax_main.plot(
        df_resampled['elapsed_min'], 
        df_resampled['speed_kmh_smooth'], 
        color=ACCENT_BLUE, 
        linewidth=2.0, 
        label='Snelheid (km/u)'
    )

    ax_main.set_xlabel('Verstreken Tijd (minuten)', color='white', fontsize=11, fontweight='bold', labelpad=8)
    ax_main.set_ylabel('Snelheid (km/u)', color=ACCENT_BLUE, fontsize=11, fontweight='bold', labelpad=8)
    ax_main.set_ylim(bottom=-0.1, top=max_speed)
    ax_main.tick_params(colors='white', labelsize=9)
    ax_main.grid(True, linestyle='--', alpha=0.15)
    ax_main.set_title('ETAPPE PERFORMANCE: BEWEEG RATIO (Actief vs. Stilstand)', color='white', fontsize=13, fontweight='bold', pad=15)

    # Stats Overlay Box
    stats_text = (
        f"Totale tijd:   {format_time(total_time_sec)}\n"
        f"In beweging:  {format_time(moving_time_sec)} ({moving_ratio:.1f}%)\n"
        f"Stilstand:    {format_time(idle_time_sec)} ({100 - moving_ratio:.1f}%)"
    )
    ax_main.text(
        0.03, 0.93, stats_text, 
        transform=ax_main.transAxes, 
        fontsize=9.5, 
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.6', facecolor=CARD_BG, edgecolor='#374151', alpha=0.85),
        color='white',
        fontfamily='monospace'
    )

    ax_main.legend(loc='upper right', facecolor=CARD_BG, edgecolor='#374151', labelcolor='white', fontsize=9)

    # --- DONUT CHART ---
    colors = [ACCENT_GREEN, ACCENT_RED]
    sizes = [moving_ratio, max(0, 100 - moving_ratio)]
    
    ax_donut.pie(
        sizes, 
        colors=colors, 
        startangle=90, 
        counterclock=False,
        wedgeprops=dict(width=0.26, edgecolor=BG_COLOR, linewidth=3)
    )

    ax_donut.text(
        0, 0, f"{int(round(moving_ratio))}%\nActief", 
        ha='center', va='center', 
        fontsize=14, fontweight='bold', color='white'
    )
    ax_donut.set_title('Actief vs. Stilstand Ratio', color='white', fontsize=11, fontweight='bold', pad=12)

    plt.tight_layout()
    return fig, df_resampled

In [0]:
# Het FIT-bestand van etappe 4
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260915_stubaier_hohenweg_etappe_4.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)
df_active = calculate_elapsed_time(df_clean)

# 2. Genereer de grafiek en haal de KPI op
fig, df_processed = create_moving_ratio_chart(
    df_active, 
    speed_threshold_kmh=2.0,  # Pas eventueel aan naar 2.5 als pauzes nog gemist worden
    min_pause_sec=45
)

# 3. Exporteer naar landscape formaat voor Instagram
export_for_instagram(fig, filename="20260915_movingratio", format_type='landscape')